In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/summer-analytics-mid-hackathon/hacktest.csv
/kaggle/input/summer-analytics-mid-hackathon/hacktrain.csv


In [2]:
df = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktrain.csv")

In [3]:
df.isnull().sum()

Unnamed: 0       0
ID               0
class            0
20150720_N     560
20150602_N    1200
20150517_N     800
20150501_N     960
20150415_N     480
20150330_N    1120
20150314_N     720
20150226_N    1360
20150210_N     640
20150125_N    1040
20150109_N     880
20141117_N    1280
20141101_N     400
20141016_N    1440
20140930_N     800
20140813_N     560
20140626_N    1600
20140610_N     480
20140525_N     720
20140509_N     880
20140423_N    1760
20140407_N     640
20140322_N    1120
20140218_N    1440
20140202_N     560
20140117_N    1200
20140101_N     400
dtype: int64

In [4]:
df.fillna(df.mean(numeric_only=True), inplace=True) #simple mean imputation [This part has a lot of scope for imporovement.]
#keep in mind that the data is inherently noisy and the test dataset is not.
df.isnull().sum()

Unnamed: 0    0
ID            0
class         0
20150720_N    0
20150602_N    0
20150517_N    0
20150501_N    0
20150415_N    0
20150330_N    0
20150314_N    0
20150226_N    0
20150210_N    0
20150125_N    0
20150109_N    0
20141117_N    0
20141101_N    0
20141016_N    0
20140930_N    0
20140813_N    0
20140626_N    0
20140610_N    0
20140525_N    0
20140509_N    0
20140423_N    0
20140407_N    0
20140322_N    0
20140218_N    0
20140202_N    0
20140117_N    0
20140101_N    0
dtype: int64

In [5]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report

# Drop ID column
df.drop(columns=['ID'], inplace=True)

# Encode class column
label_encoder = LabelEncoder()
df['class'] = label_encoder.fit_transform(df['class'])

# Split into features and target
X = df.drop(columns=['class'])
y = df['class']

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Fit logistic regression with higher iterations
model = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,  # increased from 10
    random_state=42
)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    labels=list(range(len(label_encoder.classes_))),
    target_names=label_encoder.classes_,
    zero_division=0  # Avoid undefined metric warnings
))


              precision    recall  f1-score   support

        farm       0.85      0.88      0.86       168
      forest       1.00      0.99      0.99      1232
       grass       0.83      0.64      0.72        39
  impervious       0.82      0.89      0.85       134
     orchard       1.00      0.50      0.67         6
       water       0.89      0.81      0.85        21

    accuracy                           0.96      1600
   macro avg       0.90      0.78      0.83      1600
weighted avg       0.96      0.96      0.96      1600



In [6]:
test_data = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktest.csv")
test_data.shape

(2845, 29)

In [7]:
ID=test_data['ID']
test_data.drop(['ID'],axis=1,inplace=True)

In [8]:
test_data_scaled = scaler.transform(test_data)
y_test = model.predict(test_data_scaled)

In [9]:
y_test

array([1, 1, 1, ..., 5, 5, 5])

In [10]:
y_decoded = label_encoder.inverse_transform(y_test)
y_decoded

array(['forest', 'forest', 'forest', ..., 'water', 'water', 'water'],
      dtype=object)

In [11]:
result = pd.DataFrame({
    'ID': ID,
    'class': y_decoded
})

In [12]:
result

,ID,class
0,1,forest
1,2,forest
2,3,forest
3,4,forest
4,5,forest
...,...,...
2840,2841,water
2841,2842,water
2842,2843,water
2843,2844,water


In [13]:
# result.to_csv("submission.csv", index=False) 
# this file will appear under the output section of the right navbar. You need to submit this csv file

In [14]:
# result.to_csv("submission.csv", index=False)

# Changes after 1st submission

In [15]:
# # Better Code - Score : 0.71394
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler
# from sklearn.impute import SimpleImputer
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import GridSearchCV

# # Feature engineering functions
# def extract_temporal_features(df):
#     features = pd.DataFrame(index=df.index)
    
#     # Basic statistics
#     features['mean_ndvi'] = df.mean(axis=1)
#     features['std_ndvi'] = df.std(axis=1)
#     features['min_ndvi'] = df.min(axis=1)
#     features['max_ndvi'] = df.max(axis=1)
#     features['amplitude'] = features['max_ndvi'] - features['min_ndvi']
    
#     # Seasonal features (assuming columns are sorted chronologically)
#     n = len(df.columns)
#     features['winter_avg'] = df.iloc[:, :n//4].mean(axis=1)  # first quarter
#     features['spring_avg'] = df.iloc[:, n//4:n//2].mean(axis=1)
#     # Add more seasonal splits as needed
    
#     return features

# # Main pipeline
# def build_model():
#     pipeline = Pipeline([
#         ('imputer', SimpleImputer(strategy='median')),
#         ('scaler', StandardScaler()),
#         ('clf', LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000))
#     ])
    
#     params = {
#         'clf__C': [0.1, 1, 10],
#         'clf__penalty': ['l2']  # l1 might work better with feature selection
#     }
    
#     return GridSearchCV(pipeline, params, cv=5, scoring='accuracy')

# # Load data
# train_data = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktrain.csv")
# test_data = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktest.csv")

# # Prepare features
# ndvi_cols = [col for col in train_data.columns if '_N' in col]
# X_train = extract_temporal_features(train_data[ndvi_cols])
# y_train = train_data['class']

# # Train model
# model = build_model()
# model.fit(X_train, y_train)

# # Predict on test set
# X_test = extract_temporal_features(test_data[ndvi_cols])
# predictions = model.predict(X_test)

# # Prepare submission
# submission = pd.DataFrame({
#     'ID': test_data['ID'],
#     'class': predictions
# })
# submission.to_csv('newsubmission.csv', index=False)

In [16]:
# #  Score : 0.72691
# import pandas as pd
# import numpy as np
# from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler, PolynomialFeatures
# from sklearn.impute import SimpleImputer
# from sklearn.pipeline import Pipeline
# from sklearn.feature_selection import SelectKBest, f_classif
# from sklearn.model_selection import GridSearchCV

# ## 1. Advanced Feature Engineering
# def create_high_performance_features(df):
#     features = pd.DataFrame(index=df.index)
#     ndvi_data = df.copy()
    
#     # Basic statistics
#     features['annual_mean'] = ndvi_data.mean(axis=1)
#     features['annual_std'] = ndvi_data.std(axis=1)
#     features['annual_amplitude'] = ndvi_data.max(axis=1) - ndvi_data.min(axis=1)
    
#     # Phenological features
#     features['max_ndvi'] = ndvi_data.max(axis=1)
#     features['min_ndvi'] = ndvi_data.min(axis=1)
    
#     # Date-based features (assuming columns are named like YYYYMMDD_N)
#     date_columns = pd.to_datetime([col.split('_')[0] for col in ndvi_data.columns], format='%Y%m%d')
#     day_of_year = date_columns.dayofyear.values
    
#     # Weighted seasonal features
#     for season, months in [('winter', [12,1,2]), ('spring', [3,4,5]), 
#                           ('summer', [6,7,8]), ('fall', [9,10,11])]:
#         season_mask = date_columns.month.isin(months)
#         features[f'{season}_mean'] = ndvi_data.iloc[:, season_mask].mean(axis=1)
#         features[f'{season}_std'] = ndvi_data.iloc[:, season_mask].std(axis=1)
    
#     # Growing season features
#     features['growing_season_avg'] = (features['spring_mean'] + features['summer_mean']) / 2
#     features['dormancy_avg'] = (features['winter_mean'] + features['fall_mean']) / 2
    
#     # Advanced metrics
#     features['variation_coeff'] = features['annual_std'] / features['annual_mean']
#     features['summer_winter_diff'] = features['summer_mean'] - features['winter_mean']
    
#     # Fourier transform features (first 3 harmonics)
#     fft_values = np.fft.fft(ndvi_data.fillna(0).values)
#     for i in range(1, 4):
#         features[f'fft_amp_{i}'] = np.abs(fft_values[:, i])
#         features[f'fft_phase_{i}'] = np.angle(fft_values[:, i])
    
#     return features

# ## 2. Optimized Modeling Pipeline
# def build_high_performance_model():
#     # Feature processing
#     feature_processor = Pipeline([
#         ('imputer', SimpleImputer(strategy='median')),
#         ('scaler', StandardScaler()),
#         ('poly', PolynomialFeatures(degree=2, interaction_only=True)),
#         ('selector', SelectKBest(f_classif, k=25))
#     ])
    
#     # Final model with optimized parameters
#     model = LogisticRegression(
#         multi_class='multinomial',
#         solver='saga',
#         penalty='elasticnet',
#         l1_ratio=0.5,
#         max_iter=2000,
#         class_weight='balanced',
#         C=0.05,
#         random_state=42
#     )
    
#     return Pipeline([
#         ('features', feature_processor),
#         ('model', model)
#     ])

# ## 3. Complete Implementation
# def main():
#     # Load data
#     train_data = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktrain.csv")
#     test_data = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktest.csv")

#     # Prepare features
#     ndvi_cols = [col for col in train_data.columns if '_N' in col]
#     X_train = create_high_performance_features(train_data[ndvi_cols])
#     y_train = train_data['class']
    
#     # Train model
#     model = build_high_performance_model()
#     model.fit(X_train, y_train)
    
#     # Predict on test set
#     X_test = create_high_performance_features(test_data[ndvi_cols])
#     predictions = model.predict(X_test)
    
#     # Prepare submission
#     submission = pd.DataFrame({
#         'ID': test_data['ID'],
#         'class': predictions
#     })
#     submission.to_csv('mysubmission.csv', index=False)

# if __name__ == '__main__':
#     main()

# Day 2 - Most Powerful Trained Data - Score : 0.92180

In [17]:
import pandas as pd
import numpy as np
# Load data
train_df = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktrain.csv")
test_df  = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktest.csv")

# Separate features/labels
X = train_df.drop(['ID','class'], axis=1)
y = train_df['class']
ids = test_df['ID']
X_test = test_df.drop('ID', axis=1)

# Identify NDVI columns (they end with '_N')
ndvi_cols = [col for col in X.columns if col.endswith('_N')]

# Sort NDVI columns by actual date
from datetime import datetime
dates = [datetime.strptime(col.split('_')[0], "%Y%m%d") for col in ndvi_cols]
sorted_idx = np.argsort(dates)
ndvi_sorted = [ndvi_cols[i] for i in sorted_idx]

# Reorder columns in chronological order
X = X[ndvi_sorted]
X_test = X_test[ndvi_sorted]

# Fill missing NDVI values with column (date) mean
X = X.fillna(X.mean())
X_test = X_test.fillna(X_test.mean())


In [18]:
# Add statistical features to training and test data
for df in [X, X_test]:
    df['ndvi_max']    = df.max(axis=1)
    df['ndvi_min']    = df.min(axis=1)
    df['ndvi_mean']   = df.mean(axis=1)
    df['ndvi_std']    = df.std(axis=1)
    df['ndvi_range']  = df['ndvi_max'] - df['ndvi_min']
    df['ndvi_delta']  = df.iloc[:, -1] - df.iloc[:, 0]  # last minus first NDVI

In [19]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

# Encode class labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Initialize XGBoost for multiclass classification
xgb_clf = XGBClassifier(
    n_estimators=200, learning_rate=0.1, max_depth=6, 
    subsample=0.8, colsample_bytree=0.8, random_state=42,
    objective='multi:softprob', eval_metric='mlogloss'
)

# (Optional) cross-validate to estimate accuracy
scores = cross_val_score(xgb_clf, X, y_enc, cv=5, scoring='accuracy')
print("CV mean accuracy:", scores.mean())

# Fit on full training data
xgb_clf.fit(X, y_enc)


CV mean accuracy: 0.93225


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, objective='multi:softprob', ...)

In [20]:
# Predict on test data
y_pred = xgb_clf.predict(X_test)
final_preds = le.inverse_transform(y_pred)

# Create submission DataFrame
submission = pd.DataFrame({
    'ID': ids,
    'class': final_preds
})
submission.to_csv('submission.csv', index=False)

# Let's try something advanced (Improved Codes)

In [21]:
# import pandas as pd
# import numpy as np
# from scipy.signal import medfilt
# from sklearn.preprocessing import LabelEncoder
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.model_selection import cross_val_score

# # --- 1. Load Data ---
# # Make sure to adjust the file paths if they are different for you
# try:
#     train_data  = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktrain.csv")
#     test_data  = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktest.csv")
# except FileNotFoundError:
#     print("Ensure 'hacktrain.csv' and 'hacktest.csv' are in the correct directory.")
#     # As a fallback for environments like Kaggle notebooks:
#     # train_data = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktrain.csv")
#     # test_data = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktest.csv")


# # Separate features, target, and test IDs
# X = train_data.drop(columns=['ID', 'class'])
# y = train_data['class']
# X_test = test_data.drop(columns=['ID'])
# test_ids = test_data['ID']

# # --- 2. Preprocessing ---

# # Get NDVI columns and sort them chronologically
# ndvi_cols = [col for col in X.columns if '_N' in col]
# # Sort by date (e.g., from 20150602_N to 20160526_N)
# ndvi_sorted = sorted(ndvi_cols, key=lambda d: int(d.split('_')[0]))

# # Reorder the dataframes to have dates in sequence
# X = X[ndvi_sorted]
# X_test = X_test[ndvi_sorted]


# # --- 3. IMPROVED Data Cleaning and Smoothing ---

# # STEP 3.1: Interpolate missing values instead of filling with the column mean.
# # This creates a smoother, more realistic time-series for each individual sample.
# print("Step 1: Interpolating missing values...")
# X = X.interpolate(method='linear', axis=1, limit_direction='both')
# X_test = X_test.interpolate(method='linear', axis=1, limit_direction='both')

# # STEP 3.2: Apply a median filter to smooth out noise.
# # This is very effective at removing sharp spikes/dips (e.g., from cloud cover)
# # while preserving the overall trend of the vegetation cycle. A kernel_size of 3 is a good start.
# print("Step 2: Applying median filter for smoothing...")
# X = pd.DataFrame(medfilt(X, kernel_size=3), index=X.index, columns=X.columns)
# X_test = pd.DataFrame(medfilt(X_test, kernel_size=3), index=X_test.index, columns=X_test.columns)


# # --- 4. Feature Engineering ---
# # This part of your code was already excellent. We are now applying it to the cleaned data.
# print("Step 3: Engineering statistical features...")
# for df in [X, X_test]:
#     df['ndvi_max']    = df.max(axis=1)
#     df['ndvi_min']    = df.min(axis=1)
#     df['ndvi_mean']   = df.mean(axis=1)
#     df['ndvi_std']    = df.std(axis=1)
#     df['ndvi_range']  = df['ndvi_max'] - df['ndvi_min']
#     # Getting the delta between the last and first time points of the cleaned data
#     df['ndvi_delta']  = df[ndvi_sorted[-1]] - df[ndvi_sorted[0]]


# # --- 5. Modeling (The Simple Model Hypothesis) ---
# # For a "clean" private test set, a simpler model like a Decision Tree can be
# # more effective than a complex one, as it's less likely to overfit the noise
# # from the training data and better at finding clear, generalizable rules.

# print("Step 4: Training a Decision Tree Classifier...")

# # Encode class labels
# le = LabelEncoder()
# y_enc = le.fit_transform(y)

# # Initialize a Decision Tree Classifier
# # We give it some depth to capture the necessary rules but prevent extreme overfitting.
# dt_clf = DecisionTreeClassifier(max_depth=12, random_state=42, min_samples_leaf=3)

# # Optional: Check its performance on the training data using cross-validation.
# # A high score here (>0.95) is a strong signal this approach is working.
# scores = cross_val_score(dt_clf, X, y_enc, cv=5, scoring='accuracy')
# print(f"5-Fold Cross-Validation Accuracy: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

# # Train the final model on all available training data
# dt_clf.fit(X, y_enc)


# # --- 6. Prediction and Submission ---
# print("Step 5: Generating predictions...")
# # Predict on the processed test data
# test_predictions_enc = dt_clf.predict(X_test)

# # Decode the predictions back to original class names (e.g., 'Water', 'Forest')
# test_predictions = le.inverse_transform(test_predictions_enc)

# # Create the submission file
# submission_df = pd.DataFrame({'ID': test_ids, 'class': test_predictions})
# submission_df.to_csv('submission.csv', index=False)

# print("\n'submission.csv' has been created successfully!")

# New Score : 0.92298

In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

print("--- Script Starting ---")

# --- 1. Load Data ---
try:
    train_data  = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktrain.csv")
    test_data  = pd.read_csv("/kaggle/input/summer-analytics-mid-hackathon/hacktest.csv")
except FileNotFoundError:
    print("Ensure 'hacktrain.csv' and 'hacktest.csv' are in the correct directory.")
    exit()

X = train_data.drop(columns=['ID', 'class'])
y = train_data['class']
X_test = test_data.drop(columns=['ID'])
test_ids = test_data['ID']

# --- 2. Preprocessing ---
print("Step 1: Sorting and cleaning data...")
ndvi_cols = [col for col in X.columns if '_N' in col]
ndvi_sorted = sorted(ndvi_cols, key=lambda d: int(d.split('_')[0]))

X = X[ndvi_sorted]
X_test = X_test[ndvi_sorted]

# Use interpolation to fill missing values. This is generally better than a column mean.
X = X.interpolate(method='linear', axis=1, limit_direction='both')
X_test = X_test.interpolate(method='linear', axis=1, limit_direction='both')

# --- 3. Advanced Feature Engineering ---
print("Step 2: Engineering features...")
for df in [X, X_test]:
    # Statistical Features (on cleaned data)
    df['ndvi_max']    = df.max(axis=1)
    df['ndvi_min']    = df.min(axis=1)
    df['ndvi_mean']   = df.mean(axis=1)
    df['ndvi_std']    = df.std(axis=1)
    df['ndvi_range']  = df['ndvi_max'] - df['ndvi_min']
    
    # NEW: Phenological Feature
    # This feature captures the TIMING of peak vegetation, which is a powerful differentiator.
    # We find the column index (from 0 to 26) of the maximum NDVI value for each sample.
    df['ndvi_peak_time'] = df[ndvi_sorted].idxmax(axis=1).apply(lambda x: ndvi_sorted.index(x))


# --- 4. Model Training with Hyperparameter Tuning ---
print("Step 3: Preparing model and running GridSearchCV...")

# Encode class labels
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Initialize XGBoost Classifier
# We will use GridSearchCV to find the best parameters for this model.
xgb_clf = XGBClassifier(
    random_state=42,
    objective='multi:softprob',
    eval_metric='mlogloss',
    use_label_encoder=False  # Suppress a deprecation warning
)

# Define the grid of hyperparameters to search.
# This is a focused grid. You can expand it, but be aware it will take longer to run.
param_grid = {
    'n_estimators': [300, 400],
    'max_depth': [5, 7],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8]
}

# Set up GridSearchCV
# cv=5 means 5-fold cross-validation.
# n_jobs=-1 uses all available CPU cores to speed up the search.
grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    verbose=2
)

# Fit the grid search to the data. This will train many models and find the best one.
grid_search.fit(X, y_enc)

# --- 5. Prediction and Submission ---
print("\n--- Grid Search Complete ---")
print(f"Best parameters found: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

# Get the best model found by the search
best_model = grid_search.best_estimator_

# Make predictions on the test set
print("Step 4: Generating final predictions...")
test_predictions_enc = best_model.predict(X_test)

# Decode predictions back to original class names
test_predictions = le.inverse_transform(test_predictions_enc)

# Create submission file
submission_df = pd.DataFrame({'ID': test_ids, 'class': test_predictions})
submission_df.to_csv('submission_xgb_tuned.csv', index=False)

print("\n'submission_xgb_tuned.csv' has been created successfully!")

--- Script Starting ---
Step 1: Sorting and cleaning data...
Step 2: Engineering features...
Step 3: Preparing model and running GridSearchCV...
Fitting 5 folds for each of 8 candidates, totalling 40 fits

--- Grid Search Complete ---
Best parameters found: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'subsample': 0.8}
Best cross-validation accuracy: 0.9317
Step 4: Generating final predictions...

'submission_xgb_tuned.csv' has been created successfully!
